## Section 1 — Chargement & Vérification Rapide

Sanity check avant de travailler : shape, aperçu, types, valeurs manquantes, doublons.
L'EDA approfondie est faite dans `01_EDA.ipynb`.

In [171]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder,
)

In [172]:
df = pd.read_csv("../../customer_churn_business_dataset.csv")
print(f"nb de lignes, nb de colonnes: {df.shape}")
df.head()

nb de lignes, nb de colonnes: (10000, 32)


,customer_id,gender,age,country,city,customer_segment,tenure_months,signup_channel,contract_type,monthly_logins,...,avg_resolution_time,complaint_type,csat_score,escalations,email_open_rate,marketing_click_rate,nps_score,survey_response,referral_count,churn
0,CUST_00001,Male,68,Bangladesh,London,SME,22,Web,Monthly,26,...,13.354360,Service,4.0,0,0.71,0.40,27,Satisfied,1,0
1,CUST_00002,Female,57,Canada,Sydney,Individual,9,Mobile,Monthly,7,...,25.140088,Billing,2.0,0,0.78,0.33,-19,Neutral,2,1
2,CUST_00003,Male,24,Germany,New York,SME,58,Web,Yearly,19,...,27.572928,Service,3.0,0,0.35,0.49,80,Neutral,1,0
3,CUST_00004,Male,49,Australia,Dhaka,Individual,19,Mobile,Yearly,34,...,26.420822,Technical,5.0,1,0.83,0.15,100,Neutral,0,0
4,CUST_00005,Male,65,Bangladesh,Delhi,Individual,52,Web,Monthly,20,...,26.674579,Technical,4.0,0,0.65,0.44,21,Unsatisfied,1,0


In [173]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 32 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   customer_id             10000 non-null  object 
 1   gender                  10000 non-null  object 
 2   age                     10000 non-null  int64  
 3   country                 10000 non-null  object 
 4   city                    10000 non-null  object 
 5   customer_segment        10000 non-null  object 
 6   tenure_months           10000 non-null  int64  
 7   signup_channel          10000 non-null  object 
 8   contract_type           10000 non-null  object 
 9   monthly_logins          10000 non-null  int64  
 10  weekly_active_days      10000 non-null  int64  
 11  avg_session_time        10000 non-null  float64
 12  features_used           10000 non-null  int64  
 13  usage_growth_rate       10000 non-null  float64
 14  last_login_days_ago     10000 non-null 

In [174]:
df.isna().sum()

customer_id                  0
gender                       0
age                          0
country                      0
city                         0
customer_segment             0
tenure_months                0
signup_channel               0
contract_type                0
monthly_logins               0
weekly_active_days           0
avg_session_time             0
features_used                0
usage_growth_rate            0
last_login_days_ago          0
monthly_fee                  0
total_revenue                0
payment_method               0
payment_failures             0
discount_applied             0
price_increase_last_3m       0
support_tickets              0
avg_resolution_time          0
complaint_type            2045
csat_score                   0
escalations                  0
email_open_rate              0
marketing_click_rate         0
nps_score                    0
survey_response              0
referral_count               0
churn                        0
dtype: i

In [175]:
df.duplicated().sum()


0

In [176]:
df['churn'].value_counts(normalize=True).mul(100).round(2)

churn
0    89.79
1    10.21
Name: proportion, dtype: float64

## Section 2 — Suppression des colonnes inutiles

- `customer_id` : identifiant unique, aucune valeur prédictive → supprimé
- `city` / `country` : trop de bruit géographique pour un modèle de churn SaaS,
  faible valeur prédictive attendue → supprimés

In [177]:
cols_to_drop = ['customer_id', 'city', 'country']

df = df.drop(columns=cols_to_drop)

print(df.shape)
df.head()

(10000, 29)


,gender,age,customer_segment,tenure_months,signup_channel,contract_type,monthly_logins,weekly_active_days,avg_session_time,features_used,...,avg_resolution_time,complaint_type,csat_score,escalations,email_open_rate,marketing_click_rate,nps_score,survey_response,referral_count,churn
0,Male,68,SME,22,Web,Monthly,26,7,11.762372,5,...,13.354360,Service,4.0,0,0.71,0.40,27,Satisfied,1,0
1,Female,57,Individual,9,Mobile,Monthly,7,5,26.846390,1,...,25.140088,Billing,2.0,0,0.78,0.33,-19,Neutral,2,1
2,Male,24,SME,58,Web,Yearly,19,5,23.380065,6,...,27.572928,Service,3.0,0,0.35,0.49,80,Neutral,1,0
3,Male,49,Individual,19,Mobile,Yearly,34,7,24.243136,2,...,26.420822,Technical,5.0,1,0.83,0.15,100,Neutral,0,0
4,Male,65,Individual,52,Web,Monthly,20,6,18.872323,2,...,26.674579,Technical,4.0,0,0.65,0.44,21,Unsatisfied,1,0


## Section 3 — Traitement des Valeurs Manquantes

Seule colonne avec des NaN : `complaint_type` (2 045 manquants, soit ~20%).

Ces NaN ne sont **pas** des données inconnues : ils correspondent à des clients
qui n'ont **jamais déposé de plainte**. On remplace donc par la catégorie
`"No_Complaint"`.

In [178]:
print("Valeurs manquantes avant :")
print(df['complaint_type'].isna().sum())
print()
print("Valeurs existantes :")
print(df['complaint_type'].value_counts())

Valeurs manquantes avant :
2045

Valeurs existantes :
complaint_type
Technical    3498
Billing      2427
Service      2030
Name: count, dtype: int64


In [179]:
df['complaint_type'] = df['complaint_type'].fillna('No_Complaint')

In [180]:
print("Valeurs manquantes après :")
print(df['complaint_type'].isna().sum())
print()
print("Distribution complaint_type :")
print(df['complaint_type'].value_counts())

Valeurs manquantes après :
0

Distribution complaint_type :
complaint_type
Technical       3498
Billing         2427
No_Complaint    2045
Service         2030
Name: count, dtype: int64


In [181]:
df.isna().sum()

gender                    0
age                       0
customer_segment          0
tenure_months             0
signup_channel            0
contract_type             0
monthly_logins            0
weekly_active_days        0
avg_session_time          0
features_used             0
usage_growth_rate         0
last_login_days_ago       0
monthly_fee               0
total_revenue             0
payment_method            0
payment_failures          0
discount_applied          0
price_increase_last_3m    0
support_tickets           0
avg_resolution_time       0
complaint_type            0
csat_score                0
escalations               0
email_open_rate           0
marketing_click_rate      0
nps_score                 0
survey_response           0
referral_count            0
churn                     0
dtype: int64

## Section 4 — Encodage & Normalisation

In [182]:
# Séparation X / y
X = df.drop(columns=['churn'])
y = df['churn']

print(X.shape)
print(y.value_counts())

(10000, 28)
churn
0    8979
1    1021
Name: count, dtype: int64


In [183]:
# train / test split stratifié
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print()
print("Distribution churn — train :")
print(y_train.value_counts(normalize=True).mul(100).round(2))
print()
print("Distribution churn — test :")
print(y_test.value_counts(normalize=True).mul(100).round(2))

X_train : (8000, 28)
X_test  : (2000, 28)

Distribution churn — train :
churn
0    89.79
1    10.21
Name: proportion, dtype: float64

Distribution churn — test :
churn
0    89.8
1    10.2
Name: proportion, dtype: float64


In [184]:
# Encodage et Normalisation

# 2 valeurs → OrdinalEncoder (0/1), pas besoin de créer 2 colonnes
binary_cols = ['gender', 'discount_applied', 'price_increase_last_3m']

# 3+ valeurs → OneHotEncoder
nominal_cols = [
    'customer_segment',
    'signup_channel',
    'contract_type',
    'payment_method',
    'complaint_type',
    'survey_response',
]

# Numériques → StandardScaler
numerical_cols = [col for col in X.columns if col not in binary_cols + nominal_cols]

print("Binaires  :", binary_cols)
print("Nominales :", nominal_cols)
print("Numériques:", numerical_cols)

preprocessor = ColumnTransformer(transformers=[
    ('bin', OrdinalEncoder(), binary_cols),
    ('nom', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), nominal_cols),
    ('num', StandardScaler(), numerical_cols),
])

Binaires  : ['gender', 'discount_applied', 'price_increase_last_3m']
Nominales : ['customer_segment', 'signup_channel', 'contract_type', 'payment_method', 'complaint_type', 'survey_response']
Numériques: ['age', 'tenure_months', 'monthly_logins', 'weekly_active_days', 'avg_session_time', 'features_used', 'usage_growth_rate', 'last_login_days_ago', 'monthly_fee', 'total_revenue', 'payment_failures', 'support_tickets', 'avg_resolution_time', 'csat_score', 'escalations', 'email_open_rate', 'marketing_click_rate', 'nps_score', 'referral_count']


In [185]:
X_train_processed = preprocessor.fit_transform(X_train)  # apprend ET transforme
X_test_processed  = preprocessor.transform(X_test)        # transforme seulement

print(f"X_train_processed : {X_train_processed.shape}")
print(f"X_test_processed  : {X_test_processed.shape}")

feature_names = preprocessor.get_feature_names_out()
print(f"Nombre de features : {len(feature_names)}")
# zéro NaN
print("NaN X_train :", np.isnan(X_train_processed).sum())
print("NaN X_test  :", np.isnan(X_test_processed).sum())

X_train_processed : (8000, 41)
X_test_processed  : (2000, 41)
Nombre de features : 41
NaN X_train : 0
NaN X_test  : 0


In [186]:
# Convertir en DataFrame avec les noms de colonnes
feature_names = preprocessor.get_feature_names_out()

X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_df  = pd.DataFrame(X_test_processed,  columns=feature_names)

X_train_df.head()

# Sauvegarde
DATA_PROC_PATH = Path("../data/processed")
DATA_PROC_PATH.mkdir(parents=True, exist_ok=True)

X_train_df.to_csv(DATA_PROC_PATH / 'X_train.csv', index=False)
X_test_df.to_csv(DATA_PROC_PATH / 'X_test.csv',  index=False)
y_train.to_csv(DATA_PROC_PATH / 'y_train.csv',  index=False)
y_test.to_csv(DATA_PROC_PATH / 'y_test.csv',   index=False)

print("Fichiers sauvegardés :")
print(f"  X_train : {X_train_df.shape}")
print(f"  X_test  : {X_test_df.shape}")

Fichiers sauvegardés :
  X_train : (8000, 41)
  X_test  : (2000, 41)


In [187]:
X_train_df.head()

,bin__gender,bin__discount_applied,bin__price_increase_last_3m,nom__customer_segment_Enterprise,nom__customer_segment_Individual,nom__customer_segment_SME,nom__signup_channel_Mobile,nom__signup_channel_Referral,nom__signup_channel_Web,nom__contract_type_Monthly,...,num__total_revenue,num__payment_failures,num__support_tickets,num__avg_resolution_time,num__csat_score,num__escalations,num__email_open_rate,num__marketing_click_rate,num__nps_score,num__referral_count
0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,...,-0.807949,-0.706409,-1.093234,0.174283,0.526189,-0.537129,-0.869705,-0.316064,1.001303,-0.995846
1,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,-0.504689,0.702534,-0.188894,0.375954,0.526189,-0.537129,-1.259120,-1.740576,0.358536,0.009171
2,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,-0.553602,0.702534,-0.188894,-0.358207,-0.498273,-0.537129,-0.177411,1.607028,-0.335653,-0.995846
3,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,...,3.075726,3.520419,-0.188894,-0.368010,-0.498273,-0.537129,-0.826437,0.894772,0.101429,-0.995846
4,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,...,-0.054692,0.702534,-0.188894,1.680560,-1.522735,1.301566,-1.648536,0.681095,0.564221,2.019204


In [188]:
X_test_df.head()

,bin__gender,bin__discount_applied,bin__price_increase_last_3m,nom__customer_segment_Enterprise,nom__customer_segment_Individual,nom__customer_segment_SME,nom__signup_channel_Mobile,nom__signup_channel_Referral,nom__signup_channel_Web,nom__contract_type_Monthly,...,num__total_revenue,num__payment_failures,num__support_tickets,num__avg_resolution_time,num__csat_score,num__escalations,num__email_open_rate,num__marketing_click_rate,num__nps_score,num__referral_count
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,...,0.874651,-0.706409,-0.188894,1.033845,-2.547197,-0.537129,-1.605267,0.823546,-0.309942,-0.995846
1,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,1.559430,-0.706409,-0.188894,-0.083623,0.526189,1.301566,1.553324,-0.743417,1.489806,0.009171
2,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,...,1.803994,3.520419,-0.188894,0.413115,-0.498273,-0.537129,-1.345657,-0.173612,-0.207100,1.014187
3,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,-0.749253,-0.706409,0.715446,1.008889,-0.498273,-0.537129,1.596592,1.678254,-0.001414,0.009171
4,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,...,-0.485124,2.111477,0.715446,-0.247786,-1.522735,1.301566,1.077372,-0.102387,1.309831,0.009171
